# User-User: Cosine

## A. Construya un modelo colaborativo basado en perfiles de usuario con la primera parte de los datos de ratings.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from surprise import Reader
from surprise import Dataset
from surprise.model_selection import train_test_split
from surprise import KNNBasic
from surprise import accuracy
import random

#Para garantizar reproducibilidad en resultados
seed = 10
random.seed(seed)
np.random.seed(seed)


In [2]:
train_df = pd.read_csv("ratings_train.csv")
test_df = pd.read_csv("ratings_test.csv")

train_df.head()

,userId,movieId,rating
0,50069,318,4.5
1,97443,1084,4.0
2,3743,778,4.0
3,39644,4306,5.0
4,38315,1265,4.0


In [3]:
train_df.shape, test_df.shape

((16000210, 3), (4000053, 3))

In [4]:
print("ratings:", len(train_df))
print("usuarios:", train_df['userId'].nunique())

ratings: 16000210
usuarios: 138493


Quedarnos con usuarios que tengan mayor numero de ratings por temas de memoria

In [5]:
user_counts = train_df['userId'].value_counts()

top_users = user_counts.head(6000).index

train_df = train_df[train_df['userId'].isin(top_users)]
test_df = test_df[test_df['userId'].isin(top_users)]

In [6]:
train_df = train_df.sample(1000000, random_state=42)

In [7]:
print("ratings:", len(train_df))
print("usuarios:", train_df['userId'].nunique())

ratings: 1000000
usuarios: 6000


### Train

In [8]:
#TRAIN
reader = Reader(rating_scale=(0.5,5))

train_data = Dataset.load_from_df(
    train_df[['userId','movieId','rating']],
    reader
) #Pasar a dataframe a surprise 

trainset = train_data.build_full_trainset() #Conjunto de entrenamiento con todos los datos completos sin particion


### Test

In [9]:
#TEST
testset = test_df[['userId','movieId','rating']].values.tolist()

### Modelo

In [10]:
# se crea el mismo modelo que el del ejemplo
sim_options = {'name': 'cosine',
               'user_based': True  # calcule similitud usuario-usuario
               }
algo = KNNBasic(k=20, min_k=2, sim_options=sim_options)
algo.fit(trainset) # Entrenamos con todo el dataset.por que quiero que el modelo se entrene con toda la info disponible para generar las mejores predicciones posibles.
predictions = algo.test(testset)



Computing the cosine similarity matrix...
Done computing similarity matrix.


In [11]:
print(predictions[:5])


[Prediction(uid=49018.0, iid=32.0, r_ui=2.0, est=3.75, details={'actual_k': 20, 'was_impossible': False}), Prediction(uid=40721.0, iid=32213.0, r_ui=4.0, est=2.7963520943406497, details={'actual_k': 20, 'was_impossible': False}), Prediction(uid=136187.0, iid=1816.0, r_ui=2.0, est=2.7994661414227986, details={'actual_k': 20, 'was_impossible': False}), Prediction(uid=66226.0, iid=2302.0, r_ui=3.0, est=3.9252155124167722, details={'actual_k': 20, 'was_impossible': False}), Prediction(uid=61740.0, iid=8866.0, r_ui=3.5, est=3.0260187604576343, details={'actual_k': 20, 'was_impossible': False})]


## C. Compare su predicción de rating con el efectivamente encontrado en el dataset. Establezca una forma de evaluar globalmente sus distancias en las predicciones que refleje la calidad de las mismas

In [12]:
#--------------------------------------------------------------------
#RMSE
#--------------------------------------------------------------------
print("RMSE:", accuracy.rmse(predictions))

#--------------------------------------------------------------------
#MAE
#--------------------------------------------------------------------
print("MAE:", accuracy.mae(predictions))

#--------------------------------------------------------------------
#Precision, Recall, F1
#--------------------------------------------------------------------
from collections import defaultdict

# threshold de relevancia
threshold = 4

# diccionario usuario -> lista de predicciones
user_est_true = defaultdict(list)

for uid, iid, true_r, est, _ in predictions:
    user_est_true[uid].append((est, true_r))

precisions = dict()
recalls = dict()

for uid, user_ratings in user_est_true.items():
    
    # ordenar por predicción (descendente)
    user_ratings.sort(key=lambda x: x[0], reverse=True)
    
    # top-N (puedes cambiar 10)
    top_n = user_ratings[:10]
    
    # relevantes reales
    n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
    
    # relevantes recomendados
    n_rec_k = sum((est >= threshold) for (est, _) in top_n)
    
    # relevantes correctos
    n_rel_and_rec_k = sum(
        ((true_r >= threshold) and (est >= threshold))
        for (est, true_r) in top_n
    )
    precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
    recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

#-----------------------------------
# Precision
#-----------------------------------
precision = sum(precisions.values()) / len(precisions)

#-----------------------------------
# Recall
#-----------------------------------
recall = sum(recalls.values()) / len(recalls)

#-----------------------------------
# F1 Score
#-----------------------------------
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

RMSE: 0.9176
RMSE: 0.9176424061715622
MAE:  0.7083
MAE: 0.708307443966141
Precision: 0.7085527116402186
Recall: 0.10610298760933569
F1: 0.18456768829574832


R//: 

## D. Varíe la estrategia de selección de vecinos por umbral de similitud y por número de vecinos. Revise cuál es el impacto al variar estos parámetros.

In [13]:
from surprise.model_selection import GridSearchCV
from surprise import KNNBasic

param_grid = {
    'k': [10, 20, 40],
    'min_k': [1, 3, 5],
    'sim_options': {
        'name': ['cosine'],
        'user_based': [True]
    }
}

gs = GridSearchCV(
    KNNBasic,
    param_grid,
    measures=['rmse', 'mae'],
    cv=3
)

gs.fit(train_data)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing th

In [15]:
print("Mejor RMSE:", gs.best_score['rmse'])
print("Mejores parámetros:", gs.best_params['rmse'])

Mejor RMSE: 0.9168672329823271
Mejores parámetros: {'k': 40, 'min_k': 3, 'sim_options': {'name': 'cosine', 'user_based': True}}


## E Revise la estrategia de ponderación por significancia de McLaughlin’s [1] (McLaughlin’s significance weighting) y revise cuál es el impacto al variar los parámetros de esta estrategia. 

In [ ]:
#Mclaughlin busca ajustar la simlitud segun cantidad de items compartidos. Surprise por defecto no puede hacer eso tonces toca hacerlo manual.
def filter_users(df, min_interactions):
    user_counts = df['userId'].value_counts()
    active_users = user_counts[user_counts >= min_interactions].index
    return df[df['userId'].isin(active_users)]

In [ ]:


thresholds = [10, 20, 50]  
results = []

reader = Reader(rating_scale=(0.5, 5))
#Entrenar y evaluar el modelo para cada threshold
for t in thresholds:
    print(f"\n Threshold: {t}")
    
    # 1. aplicar "significance weighting"
    df_filtered = filter_users(train_df, t)

    print("ratings:", len(df_filtered))
    print("usuarios:", df_filtered['userId'].nunique())

    # 2. convertir a formato Surprise
    data = Dataset.load_from_df(
        df_filtered[['userId', 'movieId', 'rating']],
        reader
    )

    trainset = data.build_full_trainset()

    # 3. modelo USER-USER COSINE
    algo = KNNBasic(
        k=20,
        min_k=2,
        sim_options={'name': 'cosine', 'user_based': True}
    )

    algo.fit(trainset)

    # 4. evaluar en el MISMO testset 
    predictions = algo.test(testset)

    rmse = accuracy.rmse(predictions, verbose=False)

    print(f"RMSE: {rmse}")

    results.append((t, rmse))


 Threshold: 10
ratings: 1000000
usuarios: 6000
Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9176424061715622

 Threshold: 20
ratings: 1000000
usuarios: 6000
Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9176424061715622

 Threshold: 50
ratings: 1000000
usuarios: 6000
Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.9176424061715622


In [18]:
print("\n Resultados finales:")
for t, rmse in results:
    print(f"Threshold {t}: RMSE = {rmse}")


 Resultados finales:
Threshold 10: RMSE = 0.9176424061715622
Threshold 20: RMSE = 0.9176424061715622
Threshold 50: RMSE = 0.9176424061715622
